In [202]:
import random
import pandas as pd
import numpy as np
import copy
from collections import Counter, defaultdict

In [203]:
POPULASI = 1000
VIOLATION_COST = 100
ITERATION = 1000
MUTATION_PROB = 0.7
TOURNAMENT_SIZE = 10

In [204]:
guru_df = pd.read_csv('../dataset/guru.csv')
kelas_df = pd.read_csv('../dataset/kelas.csv')
mapel_df = pd.read_csv('../dataset/mapel.csv')
relasi_guru_mapel_df = pd.read_csv('../dataset/relasi_guru_mapel.csv')
slot_df = pd.read_csv('../dataset/slot.csv')
wali_kelas_df = pd.read_csv('../dataset/wali_kelas.csv')

# DICT

In [205]:
# =========================================================
# Hari
hariId = {
"Senin": 1,
"Selasa": 2,
"Rabu": 3,
"Kamis": 4,
"Jumat": 5,
}
# hariId

# ========================================================
# mencari slot tiap per hari
# key = hariId, value = jumlah slot integer
# contoh output: {1: 8, 2: 8, 3: 8, 4: 7, 5: 5}

slotPerHari = (
    slot_df.groupby("hari")
    .size()
    .rename(index=hariId)
    .to_dict()
)
# print(slotPerHari)

slotHari = []

for hari, jumlah in slotPerHari.items():
    slotHari += [hari] * jumlah

slotHari = np.array(slotHari)

slotHariFull = np.tile(slotHari,27)

slotIndex = np.arange(972) % 36

# ========================================================
# guru dan nama
# key = guru_id, value = nama guru
guruPengajar = dict(
    zip(guru_df['guru_id'], guru_df['nama_guru'])
)
# guruPengajar

# ========================================================
# mapping nama kelas dan tingkatan
# ada 27 kelas, contoh output: {1: [{'tingkatan': 7, 'nama_kelas': '7A'}], 2: [{'tingkatan': 7, 'nama_kelas': '7B'}]}
kelasDanTingkatan = dict(
    zip(
        kelas_df["kelas_id"],
        kelas_df["tingkatan"]
    )
)
# print(kelasDanTingkatan)

# ========================================================
# mapping nama mapel dan id
# key = mapel_id, value = nama mapel
namaMapelDanId = dict(
    zip(mapel_df['mapel_id'], mapel_df['nama_mapel'])
)
# namaMapelDanId

# =========================================================
# mencari jam per minggu tiap mape
# key = mapel_id, value = jam per minggu integerl
jamPerMingguMapel = dict(
    zip(mapel_df['mapel_id'], mapel_df['jam_per_minggu'])
)
# print(jamPerMingguMapel)

# =========================================================
# mapel id dan hari MGMP
# contoh output: {1: 1, 2: 2, 3: 2, 4: 4, 5: 4, 6: 3, 7: 1, 8: 1, 9: 3, 10: 4, 11: 5, 12: 3, 13: 2}
mgmpMapel = dict(
    zip(mapel_df['mapel_id'], mapel_df['MGMP'])
)
mgmpMapel = {mapel_id: hariId[hari] for mapel_id, hari in mgmpMapel.items()}
# print(mgmpMapel)
# =========================================================
# batas siang dan batas MGMP
# key = hariId value = slot ke berapa dalam hari tersebut
batasSiang = {1: 5, 2: 5, 3: 4, 4: 5, 5: 4}
batasMGMP = {1: 2, 2: 2, 3: 2, 4: 2, 5: 1}
# =========================================================
# durasi guru mengajar
# key = guru value = list of dict {mapel_id, tingkatan, durasi}
durasiGuruMengajar = defaultdict(list)

for row in relasi_guru_mapel_df.itertuples():

    durasiGuruMengajar[row.guru_id].append({
        "mapel_id": row.mapel_id,
        "tingkatan": row.tingkatan,
        "durasi": row.durasi
    })

durasiGuruMengajar = dict(durasiGuruMengajar)
# durasiGuruMengajar


# =========================================================
# Wali kelas guru
# key = guru_id, value = kelas_id
waliKelas = dict(
    zip(wali_kelas_df['guru_id'], wali_kelas_df['kelas_id'])
)
# waliKelas

# =========================================================
# indexKelas
kelasIndex = {}
slotRange = 0
for i in range(1, 28):
    end = slotRange + 36
    kelasIndex[i] = (slotRange, end)
    slotRange = end

kelasIndex

{1: (0, 36),
 2: (36, 72),
 3: (72, 108),
 4: (108, 144),
 5: (144, 180),
 6: (180, 216),
 7: (216, 252),
 8: (252, 288),
 9: (288, 324),
 10: (324, 360),
 11: (360, 396),
 12: (396, 432),
 13: (432, 468),
 14: (468, 504),
 15: (504, 540),
 16: (540, 576),
 17: (576, 612),
 18: (612, 648),
 19: (648, 684),
 20: (684, 720),
 21: (720, 756),
 22: (756, 792),
 23: (792, 828),
 24: (828, 864),
 25: (864, 900),
 26: (900, 936),
 27: (936, 972)}

# INDIVIDU

In [206]:
def normalizeIndividu(individu):
    hasil = []
    for x in individu:
        if isinstance(x, tuple):
            hasil.append(x)
        else:
            hasil.append((int(x[0]), int(x[1])))
    return hasil

In [207]:
def blokDistribusi(jam):

    mapping = {
        2: [2],
        3: [3],
        4: [2,2],
        5: [2,3]
    }

    return mapping.get(jam, [jam])

In [208]:
def ambilGuruValid(mapel_id, tingkatan):

    guruMapelIndex = defaultdict(list)

    for guru_id, relasiList in durasiGuruMengajar.items():

        for r in relasiList:

            key = (r["mapel_id"], r["tingkatan"])

            guruMapelIndex[key].append(guru_id)
            
    return guruMapelIndex.get((mapel_id, tingkatan), [])
# # mengambil guru berdsakan mapel dan tingaktan
# def ambilGuruValid(mapel_id, tingkatan):
#     listGuru = []

#     for guru_id, relasiList in durasiGuruMengajar.items():
#         for relasi in relasiList:
#             if relasi["mapel_id"] == mapel_id and relasi["tingkatan"] == tingkatan:
#                 listGuru.append(guru_id)
#     return listGuru

In [209]:
# # membuat jadwal kosongan dulu
# def jadwalKosongan():
#     jadwal = {}
#     for hari in slotPerHari.keys():
#         jadwal[hari] = []

#     return jadwal

In [210]:
# def slotTersedia(jadwal_kelas, hari, durasi):
#     slotTerpakai = 0

#     for event in jadwal_kelas[hari]:
#         slotTerpakai += event["durasi"]
    
#     if slotTerpakai + durasi <= slotPerHari[hari]:
#         return True
    
#     return False

In [211]:
# def putEvent(jadwal_kelas, event):

#     listHari = list(slotPerHari.keys())

#     random.shuffle(listHari)

#     for hari in listHari:

#         if slotTersedia(jadwal_kelas, hari, event["durasi"]):
#             jadwal_kelas[hari].append(event)
            
#             return True
        
#     return False

In [212]:
def perluasBlok(mapel_id, guru_id, durasi):

    return [(mapel_id, guru_id)] * durasi

In [213]:
def generatePerKelas(tingkatan):

    pilihan = []

    for mapel_id, jam in jamPerMingguMapel.items():

        blok = blokDistribusi(jam)

        guruValid = ambilGuruValid(mapel_id, tingkatan)

        if not guruValid:
            continue

        guru = random.choice(guruValid)

        for durasi in blok:

            pilihan.extend(perluasBlok(mapel_id, guru, durasi))

    random.shuffle(pilihan)

    return pilihan

In [214]:
def individuConstruct(kelas_id):

    tingkatan = kelasDanTingkatan[kelas_id]

    slots = generatePerKelas(tingkatan)

    return slots

In [215]:
def individuTrigger():

    individu = []

    for kelas_id in sorted(kelasDanTingkatan.keys()):

        slots = individuConstruct(kelas_id)

        individu.extend(slots)

    return individu

In [216]:
def populasiConstruct(POPULASI):

    populasi = []

    for _ in range(POPULASI):

        individu = individuTrigger()

        individu = np.array(individu, dtype=np.int16)

        populasi.append(individu)

    return populasi

In [217]:
populasiOptimasi = populasiConstruct(POPULASI)

In [218]:
populasiOptimasi

[array([[ 5, 18],
        [ 2, 33],
        [ 3, 29],
        ...,
        [ 1, 26],
        [ 6,  9],
        [ 4, 11]], dtype=int16),
 array([[ 7, 21],
        [10, 46],
        [ 4, 19],
        ...,
        [12, 54],
        [ 3, 22],
        [ 9, 35]], dtype=int16),
 array([[ 5, 18],
        [ 3, 36],
        [13, 53],
        ...,
        [ 7,  4],
        [ 3, 20],
        [13, 52]], dtype=int16),
 array([[ 6,  2],
        [ 3, 36],
        [ 2, 45],
        ...,
        [13, 52],
        [ 3, 22],
        [ 8, 23]], dtype=int16),
 array([[ 3, 36],
        [12, 51],
        [ 5, 48],
        ...,
        [ 3, 22],
        [12, 54],
        [ 7, 25]], dtype=int16),
 array([[ 2, 45],
        [ 6,  3],
        [ 6,  3],
        ...,
        [ 8, 23],
        [12, 49],
        [ 7,  7]], dtype=int16),
 array([[ 4, 47],
        [ 6, 44],
        [ 2, 39],
        ...,
        [10, 15],
        [13, 52],
        [ 2, 10]], dtype=int16),
 array([[ 3, 36],
        [13, 53],
        [ 2,

In [219]:
# start, end = kelasIndex[1]

# kelas1 = populasiOptimasi[start:end]
# kelas1

In [220]:
# def tampilkanJadwalKelas(individu, kelas_id):

#     start, end = kelasIndex[kelas_id]

#     kelas = individu[start:end]

#     for i, gen in enumerate(kelas):

#         mapel_id, guru_id = gen

#         print(
#             "Hari", slotHari[i],
#             "| Slot", i+1,
#             "|", namaMapelDanId[mapel_id],
#             "|", guruPengajar[guru_id]
#         )

In [221]:
# tampilkanJadwalKelas(populasiOptimasi[0],1)

# EVAL

In [222]:
# ### Pre Eval
slotAwalHari = {}

index = 0

for hari, jumlah in slotPerHari.items():
    slotAwalHari[hari] = index
    index += jumlah


slotKeHari = {}

index = 0

for hari, jumlah in slotPerHari.items():
    for _ in range(jumlah):
        slotKeHari[index] = hari
        index += 1

In [223]:
def guruBentrok(individu):

    pelanggaran = 0

    slotPerKelas = 36
    jumlahKelas = len(kelasIndex)

    for slot in range(slotPerKelas):

        guruSet = set()

        for kelas in range(jumlahKelas):

            index = kelas * slotPerKelas + slot

            guru = individu[index][1]

            if guru in guruSet:
                pelanggaran += 1
            else:
                guruSet.add(guru)

    return pelanggaran
# def guruBentrok(individu):
#     pelanggaran = 0

#     for hari in slotPerHari:
#         totalSlot = slotPerHari[hari]

#         for slot in range(totalSlot):
#             guruMengajar = []

#             for kelas in individu:
#                 if slot < len(individu[kelas][hari]):
#                     guru = individu[kelas][hari][slot]["guru"]
#                     guruMengajar.append(guru)

#                 if len(guruMengajar) != len(set(guruMengajar)):
#                     pelanggaran += 1
#     return pelanggaran



In [240]:
def distribusiMapel(individu):

    individu = [tuple(x) for x in individu]
    pelanggaran = 0

    for kelas_id,(start,end) in kelasIndex.items():

        distribusi = {}

        kelasSlots = normalizeIndividu(individu[start:end])

        mapelSekarang = kelasSlots[0]
        count = 1

        blok = []

        for i in range(1,len(kelasSlots)):

            mapel = kelasSlots[i]

            if mapel == mapelSekarang:
                count += 1
            else:
                blok.append((mapelSekarang,count))
                mapelSekarang = mapel
                count = 1

        blok.append((mapelSekarang,count))

        for mapel,durasi in blok:

            if mapel not in distribusi:
                distribusi[mapel] = []

            distribusi[mapel].append(durasi)

        for mapel in distribusi:

            jam = jamPerMingguMapel[mapel[0]]

            if jam == 2:
                if distribusi[mapel] != [2]:
                    pelanggaran += 1

            elif jam == 3:
                if distribusi[mapel] != [3]:
                    pelanggaran += 1

            elif jam == 4:
                if sorted(distribusi[mapel]) != [2,2]:
                    pelanggaran += 1

            elif jam == 5:
                if sorted(distribusi[mapel]) != [2,3]:
                    pelanggaran += 1

    return pelanggaran

In [225]:
def mapelSiang(individu):

    pelanggaran = 0

    for index,(mapel,guru) in enumerate(individu):

        slotDalamKelas = index % 36

        hari = slotHari[slotDalamKelas]

        slotHariIndex = slotDalamKelas - slotAwalHari[hari]

        if mapel == 8 and slotHariIndex > batasSiang[hari]:
            pelanggaran += 1

    return pelanggaran
# def mapelSiang(individu):
#     pelanggaran = 0

#     for kelas in individu:

#         for hari in individu[kelas]:
#             batas = batasSiang[hari]

#             for slot in range(len(individu[kelas][hari])):
#                 mapel = individu[kelas][hari][slot]["mapel"]

#                 # if mapel == 8 and slot >= batas:
#                 if mapel == 8 and slot > batas:

#                     pelanggaran += 1
#     return pelanggaran

In [226]:
def durasiGuru(individu):

    guru = individu[:,1]

    unique, counts = np.unique(guru, return_counts=True)

    pelanggaran = 0

    for c in counts:
        if c > 40:
            pelanggaran += c - 40

    return pelanggaran
# def durasiGuru(individu):
#     pelanggaran = 0

#     loadGuru = {}

#     for kelas in individu:
#         for hari in individu[kelas]:

#             for slot in individu[kelas][hari]:

#                 guru = slot["guru"]

#                 if guru not in loadGuru:
#                     loadGuru[guru] = 0

#                 loadGuru[guru] += 1
#     for guru in loadGuru:
#         if loadGuru[guru] > 40:
#             pelanggaran += loadGuru[guru] - 40
#     return pelanggaran

In [227]:
def waktuMGMP(individu):

    pelanggaran = 0

    for index,(mapel,guru) in enumerate(individu):

        if mapel in mgmpMapel:

            slotDalamKelas = index % 36

            hari = slotHari[slotDalamKelas]

            slotHariIndex = slotDalamKelas - slotAwalHari[hari]

            if hari == mgmpMapel[mapel]:

                if slotHariIndex > batasMGMP[hari]:

                    pelanggaran += 1

    return pelanggaran
# def waktuMGMP(individu):
#     pelanggaran = 0

#     for kelas in individu:

#         for hari in individu[kelas]:
#             for slot in range(len(individu[kelas][hari])):
#                 mapel = individu[kelas][hari][slot]["mapel"]

#                 if mapel in mgmpMapel:
#                     hariMGMP = mgmpMapel[mapel]

#                     if hari == hariMGMP:
#                         if slot > batasMGMP[hari]:
#                             pelanggaran += 1

#     return pelanggaran

In [228]:
def cekWaliKelas(individu):

    pelanggaran = 0

    for kelas_id,(start,end) in kelasIndex.items():

        for i in range(start,end):

            guru = individu[i,1]

            if guru in waliKelas:

                if kelas_id != waliKelas[guru]:

                    pelanggaran += 1

    return pelanggaran
# def cekWaliKelas(individu):
#     pelanggaran = 0

#     for kelas_id, jadwalKelas in individu.items():

#         for hari in jadwalKelas:

#             for slot in jadwalKelas[hari]:

#                 guru = slot['guru']

#                 if guru in waliKelas:

#                     kelasWali = waliKelas[guru]

#                     if kelas_id != kelasWali:
#                         pelanggaran += 1
#     return pelanggaran

In [242]:
def evaluasiIndividu(individu):

    individu = normalizeIndividu(individu)
    individu = np.array(individu)
            
    pelanggaran = 0

    pelanggaran += guruBentrok(individu)
    if pelanggaran > 50: return pelanggaran * VIOLATION_COST

    pelanggaran += distribusiMapel(individu)
    pelanggaran += mapelSiang(individu)
    pelanggaran += durasiGuru(individu)
    pelanggaran += waktuMGMP(individu)
    pelanggaran += cekWaliKelas(individu)

    return pelanggaran * VIOLATION_COST

# CACHE

In [230]:
fitnessCache = {}

def hashIndividu(individu):
    return individu.tobytes()

In [231]:
def evaluasiCache(individu):

    key = hashIndividu(individu)

    if key in fitnessCache:
        return fitnessCache[key]

    fitness = evaluasiIndividu(individu)

    fitnessCache[key] = fitness

    return fitness

# PROBLEM SLOT

In [232]:
def slotBermasalah(individu):

    masalah = []

    slotPerkelas = 36
    jumlahKelas = len(kelasIndex)

    for slot in range(slotPerkelas):

        guruMengajar = {}

        for kelas in range(jumlahKelas):

            index = kelas * slotPerkelas + slot

            mapel, guru = individu[index]

            if guru in guruMengajar:

                masalah.append(index)
            else:
                guruMengajar[guru] = index
    
    return masalah

# GA

In [233]:
def kelasBermasalah(individu):

    masalah = set(slotBermasalah(individu))

    konflik = {}

    for kelas,(start,end) in kelasIndex.items():

        konflik[kelas] = 0

        for i in range(start,end):

            if i in masalah:
                konflik[kelas] += 1

    max_k = max(konflik.values())

    kandidat = [k for k,v in konflik.items() if v == max_k]

    return random.choice(kandidat)

def crossover(parent1, parent2):

    child = parent1.copy()

    kelas = random.choice(list(kelasIndex.keys()))

    start, end = kelasIndex[kelas]

    child[start:end] = parent2[start:end]

    return child

def crossoverTargeted(parent1,parent2):

    child = parent1.copy()

    kelas = kelasBermasalah(parent1)

    start,end = kelasIndex[kelas]

    child[start:end] = parent2[start:end]

    return child

# def crossover(parent1, parent2):
#     child1 = copy.deepcopy(parent1)
#     child2 = copy.deepcopy(parent2)

#     hari = list(parent1.keys())

#     # pilih hari yang akan ditukar
#     jumlah = random.randint(1, len(hari)//2)

#     hariTerpilih = random.sample(hari, jumlah)

#     for h in hariTerpilih:
#         child1[h], child2[h] = parent2[h], parent1[h]
        
#     return child1, child2

In [234]:
def clusterMapel(slot):

    blok = []

    mapelSekarang = slot[0][0]
    current = [slot[0]]

    for mapel,guru in slot[1:]:

        if mapel == mapelSekarang:
            current.append((mapel,guru))
        else:
            blok.append(current)
            current = [(mapel,guru)]
            mapelSekarang = mapel

    blok.append(current)

    random.shuffle(blok)

    hasil = []

    for b in blok:
        hasil.extend(b)

    return hasil

def mutasi(individu, MUTATION_PROB):

    child = individu.copy()

    if random.random() > MUTATION_PROB:
        return child

    kelas = random.choice(list(kelasIndex.keys()))

    start, end = kelasIndex[kelas]

    slotKelas = child[start:end]

    slotKelas = clusterMapel(slotKelas)

    child[start:end] = slotKelas

    return child

def mutasiTargeted(individu):

    child = individu.copy()

    masalah = slotBermasalah(child)

    if not masalah:
        return child

    i = random.choice(masalah)

    kelas = i // 36
    start = kelas * 36
    end = start + 36

    j = random.randint(start,end-1)

    child[i],child[j] = child[j],child[i]

    return child

# def mutasi(individu, MUTATION_PROB):

#     individuBaru = copy.deepcopy(individu)

#     if random.random() > MUTATION_PROB:
#         return individuBaru

#     hari = random.choice(list(individuBaru.keys()))

#     slot = individuBaru[hari]

#     # jika slot adalah dict
#     if isinstance(slot, dict):
#         keys = list(slot.keys())

#         if len(keys) < 2:
#             return individuBaru

#         i, j = random.sample(keys, 2)

#         slot[i], slot[j] = slot[j], slot[i]

#     # jika slot adalah list
#     else:
#         if len(slot) < 2:
#             return individuBaru

#         i, j = random.sample(range(len(slot)), 2)

#         slot[i], slot[j] = slot[j], slot[i]

#     return individuBaru

In [235]:
def turnamen(populasi, fitnessPop, TOURNAMENT_SIZE):

    kandidat =random.sample(range(len(populasi)), TOURNAMENT_SIZE)

    terbaik = kandidat[0]

    for i in kandidat:

        if fitnessPop[i] < fitnessPop[terbaik]:
            terbaik = i
    
    return populasi[terbaik]
# def turnamen(populasi, TOURNAMENT_SIZE):
#     kandidat = random.sample(populasi, TOURNAMENT_SIZE)

#     terbaik = min(kandidat, key=lambda x: evaluasiIndividu(x))

#     return terbaik

# MAIN

In [236]:
def geneticAlgorithm(populasiAwal, ITERATION, POPULASI, TOURNAMENT_SIZE, MUTATION_PROB):

    populasi = populasiAwal

    bestIndividu = None
    bestFitness = float("inf")

    for gen in range(ITERATION):

        # =============================
        # Evaluasi fitness populasi
        # =============================
        fitnessPop = []

        for individu in populasi:

            fitness = evaluasiCache(individu)

            fitnessPop.append(fitness)

            if fitness < bestFitness:
                bestFitness = fitness
                bestIndividu = individu

        print("ITERATION:", gen, "Best Fitness:", bestFitness)

        # =============================
        # Elitism (menyimpan individu terbaik)
        # =============================
        elitIndex = sorted(range(len(fitnessPop)), key=lambda i: fitnessPop[i])

        elit = [populasi[i] for i in elitIndex[:2]]

        # =============================
        # Membuat populasi baru
        # =============================
        populasiBaru = elit.copy()

        while len(populasiBaru) < POPULASI:

            # -------------------------
            # Selection
            # -------------------------
            parent1 = turnamen(populasi, fitnessPop, TOURNAMENT_SIZE)
            parent2 = turnamen(populasi, fitnessPop, TOURNAMENT_SIZE)

            # -------------------------
            # Crossover
            # -------------------------
            if random.random() < 0.5:
                child = crossover(parent1, parent2)
            else:
                child = crossoverTargeted(parent1, parent2)

            # -------------------------
            # Mutation
            # -------------------------
            if random.random() < MUTATION_PROB:

                if random.random() < 0.5:
                    child = mutasi(child, MUTATION_PROB)
                else:
                    child = mutasiTargeted(child)

            populasiBaru.append(child)

        populasi = populasiBaru

    return bestIndividu, bestFitness

In [237]:
def main():

    # # =========================
    # # Parameter GA
    # # =========================
    # POPULASI = 50
    # GENERASI = 200
    # TOURNAMENT_SIZE = 3
    # MUTATION_PROB = 0.2

    # =========================
    # Generate populasi awal
    # =========================
    populasiAwal = populasiConstruct(POPULASI)

    print("Populasi awal berhasil dibuat:", len(populasiAwal))

    # =========================
    # Jalankan Genetic Algorithm
    # =========================
    bestIndividu, bestFitness = geneticAlgorithm(
        populasiAwal,
        ITERATION,
        POPULASI,
        TOURNAMENT_SIZE,
        MUTATION_PROB
    )

    # =========================
    # Tampilkan hasil
    # =========================
    print("\n==============================")
    print("HASIL TERBAIK")
    print("==============================")
    print("Fitness:", bestFitness)

    # tampilkanJadwal(bestIndividu)

In [243]:
if __name__ == "__main__":
    main()

Populasi awal berhasil dibuat: 1000
ITERATION: 0 Best Fitness: 18200
ITERATION: 1 Best Fitness: 17600
ITERATION: 2 Best Fitness: 17300
ITERATION: 3 Best Fitness: 16100
ITERATION: 4 Best Fitness: 15600
ITERATION: 5 Best Fitness: 15100
ITERATION: 6 Best Fitness: 14900
ITERATION: 7 Best Fitness: 14400
ITERATION: 8 Best Fitness: 14200
ITERATION: 9 Best Fitness: 13900
ITERATION: 10 Best Fitness: 13600
ITERATION: 11 Best Fitness: 12900
ITERATION: 12 Best Fitness: 12800
ITERATION: 13 Best Fitness: 12400
ITERATION: 14 Best Fitness: 12000
ITERATION: 15 Best Fitness: 11600
ITERATION: 16 Best Fitness: 11400
ITERATION: 17 Best Fitness: 11200
ITERATION: 18 Best Fitness: 11000
ITERATION: 19 Best Fitness: 10800
ITERATION: 20 Best Fitness: 10600
ITERATION: 21 Best Fitness: 10400
ITERATION: 22 Best Fitness: 10300
ITERATION: 23 Best Fitness: 10100
ITERATION: 24 Best Fitness: 9900
ITERATION: 25 Best Fitness: 9600
ITERATION: 26 Best Fitness: 9500
ITERATION: 27 Best Fitness: 9200
ITERATION: 28 Best Fitness